# Capas y representaciones

**Explorador de Hespérides · Capítulo 1**

Ampliación programada sobre los conceptos de los notebooks D2L de este capítulo.

Las dos coordenadas de cada capa se muestran completas: no hay una proyección oculta. Cada punto conserva su clase y su identidad. El ejemplo usa todos sus puntos para entrenamiento; ilustra representaciones, no estima generalización.

![Ilustración conceptual](../recursos/ilustraciones/capitulo_1.png)

*Ilustración conceptual generada con ImageGen. Los resultados cuantitativos son los del código.*

In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact

torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(2)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "animation.embed_limit": 40})


In [ ]:

# Dos medias lunas: el orden de los puntos se mantiene a través de las capas.
generador = torch.Generator().manual_seed(42)
t = torch.linspace(0, np.pi, 120)
X = torch.cat([torch.stack([torch.cos(t), torch.sin(t)], 1),
               torch.stack([1-torch.cos(t), .5-torch.sin(t)], 1)])
X += .06 * torch.randn(X.shape, generator=generador)
y = torch.cat([torch.zeros(120), torch.ones(120)])
red = nn.Sequential(nn.Linear(2, 2), nn.Tanh(), nn.Linear(2, 2),
                    nn.Tanh(), nn.Linear(2, 1))
optimizador = torch.optim.Adam(red.parameters(), lr=.025)
historia, perdidas = [], []
for epoca in range(601):
    optimizador.zero_grad()
    logits = red(X).squeeze(1)
    perdida = nn.functional.binary_cross_entropy_with_logits(logits, y)
    perdida.backward()
    optimizador.step()
    perdidas.append(perdida.item())
    if epoca % 10 == 0:
        with torch.no_grad():
            h1 = red[:2](X); h2 = red[:4](X)
            historia.append((epoca, h1.numpy().copy(), h2.numpy().copy(),
                             red[4].weight.numpy().copy(), red[4].bias.item()))

def ver_capas(paso=0):
    epoca, h1, h2, w, b = historia[paso]
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
    for ax, puntos, titulo in zip(axes, [X.numpy(), h1, h2],
                                ['Entrada', 'Capa oculta 1', 'Capa oculta 2']):
        ax.scatter(*puntos.T, c=y, cmap='coolwarm', s=13, vmin=0, vmax=1)
        ax.set(title=titulo, xlabel='Coordenada 1', ylabel='Coordenada 2')
    # En la última representación, la salida es una frontera afín.
    xx, yy = np.meshgrid(np.linspace(-1.1, 1.1, 80), np.linspace(-1.1, 1.1, 80))
    score = w[0, 0]*xx + w[0, 1]*yy + b
    if score.min() < 0 < score.max():axes[2].contour(xx, yy, score, levels=[0], colors='black')
    axes[1].set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1))
    axes[2].set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1))
    axes[3].plot(perdidas, color='#087E8B')
    axes[3].axvline(epoca, color='#D99B18')
    axes[3].set(title=f'Época {epoca}', xlabel='Época', ylabel='Pérdida de entrenamiento')
    fig.tight_layout()
    plt.show()

interact(ver_capas, paso=widgets.IntSlider(min=0, max=len(historia)-1, value=30,
                                         description='Instante', continuous_update=False));


## Vista de referencia

Esta figura conserva el estado inicial también en una exportación sin kernel. Los controles anteriores se utilizan en Jupyter.

In [ ]:
ver_capas(30)

## El proceso en movimiento

Puedes reproducir, pausar y recorrer los fotogramas. La animación se genera a partir de los estados calculados arriba.

In [ ]:

# Animación autónoma: también funciona en una exportación HTML sin kernel.
fig, ax = plt.subplots(figsize=(5, 4))
puntos = ax.scatter(*historia[0][2].T, c=y, cmap='coolwarm', s=18, vmin=0, vmax=1)
ax.set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), xlabel='Coordenada 1', ylabel='Coordenada 2')
def avanzar(i):
    puntos.set_offsets(historia[i][2])
    ax.set_title(f'Representación aprendida · época {historia[i][0]}')
    return puntos,
animacion = FuncAnimation(fig, avanzar, frames=len(historia), interval=100, blit=False)
plt.close(fig)
display(HTML(animacion.to_jshtml()))


## Comprobación

Modifica un control cada vez y describe qué cambia y qué permanece constante. Compara tu observación con las preguntas del capítulo.